In [1]:
import geopandas as gpd
import pandas as pd 

In [5]:
new_rcid = gpd.read_file("geo/postworkshopcentres.gpkg")
old_rcid = gpd.read_file("geo/old_retail_mapmaker/retail_boundaries_uk.gpkg")

In [6]:
new_rcid.rename(columns={'RC_ID': 'new_rcid'}, inplace=True)

In [ ]:
# 2) Ensure both layers share the same CRS
if new_rcid.crs != old_rcid.crs:
    old_rcid = old_rcid.to_crs(new_rcid.crs)

# 3) Optional: Reproject to a metric CRS (for correct area in m²)
# Pick an appropriate British projected CRS if your data is UK (e.g. EPSG:27700 – OSGB 1936 / British National Grid)
metric_crs = "EPSG:27700"
new_rcid_metric = new_rcid.to_crs(metric_crs)
old_rcid_metric = old_rcid.to_crs(metric_crs)

# 4) Compute intersections (this can be heavy for very large datasets)
# Using overlay 'intersection' to get the actual overlap geometries
intersections = gpd.overlay(old_rcid_metric[['RC_ID', 'geometry']],
                            new_rcid_metric[['new_rcid', 'geometry']],
                            how='intersection')

# 5) Compute overlap areas
intersections['overlap_area_m2'] = intersections.geometry.area

# 6) For each old RC_ID, pick the new_rcid with the largest overlap
best_matches = (
    intersections
    .sort_values(['RC_ID', 'overlap_area_m2'], ascending=[True, False])
    .drop_duplicates(subset=['RC_ID'])  # keep the top overlap per old RC_ID
    [['RC_ID', 'new_rcid', 'overlap_area_m2']]
)

# 7) Build lookup table (1 old → 1 best new)
lookup = best_matches[['RC_ID', 'new_rcid']].copy()



/opt/anaconda3/envs/geods/lib/python3.13/site-packages/shapely/set_operations.py:168: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


In [8]:
lookup

,RC_ID,new_rcid
1669,RC_EW_1,2327
1670,RC_EW_10,5149
1671,RC_EW_100,2782
1672,RC_EW_1000,5165
1673,RC_EW_1001,3351
...,...,...
6306,RC_SC_95,1599
5748,RC_SC_96,8896
5750,RC_SC_97,3308
6190,RC_SC_98,3717


In [9]:
lookup.to_csv("output/rcid_lookup_old_to_new.csv", index=False)